### ADL_Procee_Upload.py

### This script automates the process of ADL data ETL.

### Usage:
#####     python adl_mitre.py [YYYYMMDD]
#####     If no date is provided, the script defaults to retrieving the report for the previous day.

### Author:
####     Kimia Keyvan

### Date:
####     April 2026

In [1]:
import sys
import pandas as pd
import numpy as np
import datetime as dt
from datetime import datetime, timedelta
import csv
from typing import List, Tuple
from sqlalchemy import create_engine
import cx_Oracle
import paramiko
import os
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

In [2]:
def SendEmail(repDate, status, message):
    sender_email = "..."
    receiver_email = ["..."]

    try:
        report_date = f"{repDate[4:6]}/{repDate[6:8]}/{repDate[0:4]}"
    except Exception:
        report_date = repDate

    subject_text = f"{process_name} for {report_date}"
    if status == "success":
        subject_text += " - Success"
    elif status == "error":
        subject_text += " - Failure"
    elif status == "NA":
        subject_text += " - Data not available"
    elif status == "low":
        subject_text += " - Low record counts"
        
    message_text = f"{process_name} for {report_date}\n\n{message}"

    message = MIMEMultipart("alternative")
    message["Subject"] = subject_text
    message["From"] = sender_email
    message['To'] = ", ".join(receiver_email)

    message_text_type = MIMEText(message_text, "plain")
    message.attach(message_text_type)

    smtp_server = "..."
    port = 587  

    try:
        server = smtplib.SMTP(smtp_server)
        server.ehlo()
        server.starttls()
        server.ehlo()
        server.sendmail(sender_email, receiver_email, message.as_string())
        print("Email sent.")
    except Exception as e:
        print("Error sending email:", e)
    finally:
        if server:
            server.quit()

In [3]:
def SFTPDownload(repDate):
    
    sftp_server = '...'
    sftp_username = '...'
    sftp_password = '...'
    local_dir = os.getcwd()
    local_paths = []
    
    transport = paramiko.Transport((sftp_server, 22))
    
    try:
        transport.connect(username=sftp_username, password=sftp_password)
        sftp = paramiko.SFTPClient.from_transport(transport)
        sftp.chdir(remote_directory)

        remote_files = sftp.listdir()

        prefix = f"adl_{repDate}_"
        matching_files = [f for f in remote_files if f.startswith(prefix) and f.endswith(".csv")]

        # If none found, error and exit
        if not matching_files:
            SendEmail(repDate, "error", f"No files found for {prefix}N.csv")
            sys.exit(1)

        # Sort to download in order (_1, _2, _3, ...)
        matching_files.sort()

        for fname in matching_files:
            remote_path = f"{remote_directory}/{fname}"
            local_path  = os.path.join(local_dir, fname)
            sftp.get(remote_path, local_path)
            local_paths.append(local_path)

        return local_paths

    finally:
        try:
            if sftp:
                sftp.close()
        except:
            pass
        transport.close()

In [16]:
def ConnectToOracle(repDate):
    try:
        login_info = pd.read_excel("...")
        login_info.set_index("Schema", inplace=True)

        username = "kimia"
        password = login_info.loc[username]["Pass"]
        service_name = login_info.loc[username]["Database"]
        host = "..."
        port = "1521"
        
        oracle_connection_string = (
            f"oracle+cx_oracle://{username}:{password}@{host}:{port}/"
            f"?service_name={service_name}"
        )
        engine = create_engine(oracle_connection_string)
        
        dsn = cx_Oracle.makedsn(host, port, service_name=service_name)
        connection = cx_Oracle.connect(user=username, password=password, dsn=dsn)

        return engine, connection
    
    except Exception as e:
        error_msg = f"Failed Due to Oracle Connection: {e}"
        SendEmail(repDate,"error",error_msg)  
        sys.exit(1) 
        return None, None

In [5]:
def RepairFile(repDate, file_path, table_schema):
    """
    Repairs CSV rows with extra columns caused by unquoted commas in string columns.
    Uses all Oracle VARCHAR2/string columns that exist in the CSV header.
    """

    try:

        with open(file_path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.reader(f, quotechar='"', doublequote=True, skipinitialspace=True)
            raw_header = next(reader)

        raw_header = [str(h).strip() for h in raw_header]
        expected_cols = len(raw_header)

        header_lookup = {
            str(h).strip().upper(): idx
            for idx, h in enumerate(raw_header)
        }

        string_col_indexes = []

        for oracle_col, oracle_type in table_schema.items():
            if oracle_type != "VARCHAR2":
                continue

            lookup_key = str(oracle_col).strip().upper()

            if lookup_key in header_lookup:
                string_col_indexes.append(header_lookup[lookup_key])

        if not string_col_indexes:
            return None, {
                "repair_used": False,
                "repair_skipped": True,
                "skip_reason": "No string columns found for CSV repair.",
                "expected_cols": expected_cols,
                "num_repaired": 0,
                "repair_target_columns": [],
                "repaired_line_nums": None
            }

        repaired_count = 0
        repaired_target_columns = set()

        def _repair(fields):
            nonlocal repaired_count

            fields = ["" if value is None else str(value) for value in fields]
            repaired_count += 1

            surplus = len(fields) - expected_cols

            if surplus <= 0:
                while len(fields) < expected_cols:
                    fields.append("")
                return fields

            best_repair = None
            best_score = -1
            best_idx = None

            for idx in string_col_indexes:
                if idx + surplus >= len(fields):
                    continue

                repaired = (
                    fields[:idx]
                    + [",".join(fields[idx:idx + surplus + 1])]
                    + fields[idx + surplus + 1:]
                )

                if len(repaired) != expected_cols:
                    continue

                score = 0

                for col_name, col_type in table_schema.items():
                    col_key = str(col_name).strip().upper()

                    if col_key not in header_lookup:
                        continue

                    col_idx = header_lookup[col_key]

                    if col_idx >= len(repaired):
                        continue

                    value = str(repaired[col_idx]).strip()

                    if value == "":
                        continue

                    if col_type == "NUMBER":
                        parsed_value = value.replace(",", "")
                        if not pd.isna(pd.to_numeric(parsed_value, errors="coerce")):
                            score += 1

                    elif col_type == "DATE":
                        if not pd.isna(pd.to_datetime(value, errors="coerce")):
                            score += 1

                    else:
                        score += 1

                if score > best_score:
                    best_score = score
                    best_repair = repaired
                    best_idx = idx

            if best_repair is not None:
                repaired_target_columns.add(raw_header[best_idx])
                return best_repair

            return fields[:expected_cols]

        df = pd.read_csv(
            file_path,
            dtype=str,
            engine="python",
            quotechar='"',
            doublequote=True,
            skipinitialspace=True,
            on_bad_lines=_repair,
            encoding="utf-8-sig"
        )

        df.columns = raw_header

        report = {
            "repair_used": True,
            "repair_skipped": False,
            "expected_cols": expected_cols,
            "num_repaired": repaired_count,
            "repair_target_columns": sorted(repaired_target_columns),
            "repaired_line_nums": None
        }

        return df, report

    except Exception as e:
        error_msg = f"Failed Due to File Repairing prior to Upload into Oracle: {e}"
        SendEmail(repDate, "error", error_msg)
        sys.exit(1)
        return None, None


In [6]:
def AlignDataFrameToOracleColumns(df, table_schema):
    """
    Renames CSV/DataFrame columns to exactly match Oracle column names.
    Preserves quoted mixed-case Oracle columns like "Flight Services".
    """
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    csv_lookup = {
        str(c).strip().upper(): c
        for c in df.columns
    }

    rename_map = {}

    for oracle_col in table_schema.keys():
        oracle_col = str(oracle_col).strip()

        if oracle_col in df.columns:
            continue

        lookup_key = oracle_col.upper()

        if lookup_key in csv_lookup:
            rename_map[csv_lookup[lookup_key]] = oracle_col

    return df.rename(columns=rename_map)


In [7]:
def GetTableSchema(repDate):
    try:
        engine, connection = ConnectToOracle(repDate)

        sql = """
            SELECT column_name, data_type
            FROM all_tab_columns
            WHERE table_name = :target_table
              AND owner = :target_schema
            ORDER BY column_id
        """

        type_map = {
            "NUMBER": "NUMBER",
            "FLOAT": "NUMBER",
            "BINARY_FLOAT": "NUMBER",
            "BINARY_DOUBLE": "NUMBER",
            "DECIMAL": "NUMBER",
            "INTEGER": "NUMBER",
            "INT": "NUMBER",
            "SMALLINT": "NUMBER",
            "DOUBLE PRECISION": "NUMBER",
            "REAL": "NUMBER",
            "NUMERIC": "NUMBER",

            "DATE": "DATE",
            "TIMESTAMP": "DATE",
            "TIMESTAMP WITH TIME ZONE": "DATE",
            "TIMESTAMP WITH LOCAL TIME ZONE": "DATE",

            "VARCHAR2": "VARCHAR2",
            "NVARCHAR2": "VARCHAR2",
            "CHAR": "VARCHAR2",
            "NCHAR": "VARCHAR2",
            "CLOB": "VARCHAR2",
            "NCLOB": "VARCHAR2",
            "LONG": "VARCHAR2",
            "ROWID": "VARCHAR2",
            "UROWID": "VARCHAR2",

            "RAW": "VARCHAR2",
            "XMLTYPE": "VARCHAR2",
            "BLOB": "VARCHAR2",
            "BFILE": "VARCHAR2",
        }

        binds = {
            "target_table": str(target_table).upper(),
            "target_schema": str(target_schema).upper()
        }

        with connection.cursor() as cur:
            cur.execute(sql, binds)
            rows = cur.fetchall()

        connection.close()

        if not rows:
            SendEmail(
                repDate,
                "error",
                f"No columns found for {target_schema}.{target_table}"
            )
            sys.exit(1)

        schema = {
            str(col).strip(): type_map.get((dtype or "").upper(), "VARCHAR2")
            for col, dtype in rows
        }

        return schema

    except Exception as e:
        error_msg = f"Failed Due to Table Schema: {e}"
        SendEmail(repDate, "error", error_msg)
        sys.exit(1)
        return None

In [8]:
def ProcessFile(repDate, file_path):
    """
    Reads incoming CSV file(s), cleans/types the data, and returns a DataFrame
    ready to load into the target Oracle table.
    """

    try:
        # Get target Oracle schema first because RepairFile uses it
        table_schema = GetTableSchema(repDate)

        # Read one or multiple CSV files
        if isinstance(file_path, (list, tuple)):
            all_dfs = []

            for fp in file_path:
                if os.path.exists(fp):
                    try:
                        temp_df = pd.read_csv(fp, dtype=str, encoding="utf-8-sig")
                    except pd.errors.ParserError:
                        temp_df, repair_report = RepairFile(repDate, fp, table_schema)

                        if temp_df is None:
                            temp_df = pd.read_csv(fp, dtype=str, encoding="utf-8-sig")

                    all_dfs.append(temp_df)
                else:
                    SendEmail(repDate, "error", f"File not found: {fp}")
                    sys.exit(1)

            df_raw = pd.concat(all_dfs, ignore_index=True)

        else:
            if os.path.exists(file_path):
                try:
                    df_raw = pd.read_csv(file_path, dtype=str, encoding="utf-8-sig")
                except pd.errors.ParserError:
                    df_raw, repair_report = RepairFile(repDate, file_path, table_schema)

                    if df_raw is None:
                        df_raw = pd.read_csv(file_path, dtype=str, encoding="utf-8-sig")
            else:
                SendEmail(repDate, "error", f"File not found: {file_path}")
                sys.exit(1)

        # Replace blank / placeholder values with NaN
        df_raw = df_raw.replace(r'^\s*$', np.nan, regex=True)
        df_raw = df_raw.replace(
            ['N/A', 'n/a', 'NA', 'na', 'None', 'none', None],
            np.nan
        )

        # Align CSV columns to exact Oracle column names
        df = AlignDataFrameToOracleColumns(df_raw, table_schema)

        # Coerce each column based on Oracle schema
        for col, coltype in table_schema.items():
            if col not in df.columns:
                continue
        
            if coltype == "NUMBER":
                df[col] = pd.to_numeric(
                    df[col]
                    .astype(str)
                    .str.replace(",", "", regex=False)
                    .str.strip(),
                    errors="coerce"
                )
        
            elif coltype == "DATE":
                df[col] = pd.to_datetime(
                    df[col]
                    .astype(str)
                    .str.strip(),
                    format="mixed",
                    errors="coerce"
                )
        
#            else:
#                df[col] = (
#                    df[col]
#                    .where(pd.notnull(df[col]), None)
#                    .map(lambda x: str(x).strip() if x is not None else None)
#                )
            else:
                df[col] = df[col].map(lambda x: None if pd.isna(x) else str(x).strip())
                df[col] = df[col].replace("", None)

        if date_column not in df.columns:
            SendEmail(repDate, "error", f"Date column not found in file: {date_column}")
            sys.exit(1)

        df["DATA_DATE"] = pd.to_datetime(
            df[date_column],
            errors="coerce"
        )#.dt.date

        df["LOAD_DATE"] = pd.Timestamp.now()

        # Convert pandas missing values to Python None for Oracle
        #df = df.astype(object).where(pd.notnull(df), None)
        for col in df.select_dtypes(include=["object", "string"]).columns:
            df[col] = df[col].where(pd.notnull(df[col]), None)
            
        # Row-count validation
        if len(df) < low_count_threshold:
            SendEmail(
                repDate,
                "error",
                f"Upload to Oracle Failed Due to row count in report being "
                f"less than {low_count_threshold}. Found {len(df)} rows."
            )
            sys.exit(1)

        return df

    except Exception as e:
        error_msg = f"Failed Due to File Processing for Upload into Oracle: {e}"
        SendEmail(repDate, "error", error_msg)
        sys.exit(1)
        return None

In [9]:
def UploadToOracle(repDate,dff):
    try:
        
        engine, connection = ConnectToOracle(repDate)

        # Convert all pandas missing values to real Python None before Oracle upload
        dff = dff.astype(object).where(pd.notnull(dff), None)

        repDate_dt = datetime.strptime(repDate, "%Y%m%d")
        min_date = repDate_dt - timedelta(days=0)
        max_date = repDate_dt + timedelta(days=1)

        bind_values = {"lb": min_date, "ub": max_date}
        
        # Delete existing records in range
        cursor = connection.cursor()
        delete_sql = f"DELETE FROM {target_schema}.{target_table} WHERE {date_column} >= :lb AND {date_column} < :ub"
        cursor.execute(delete_sql, {"lb": min_date, "ub": max_date})
        connection.commit()
        cursor.close()
        
        
        
        # Load in batches
        batch_size = 10000 
        chunks = np.array_split(dff, len(dff) // batch_size + 1)
        
        for chunk in chunks:
            chunk.to_sql(target_table, con=engine, schema=target_schema,if_exists='append', index=False)

        

        PROC_CALL = "BEGIN CCA.ADL_AJR_CODE(:p1); END;" 
    
        try:
            cursor.execute(PROC_CALL, (repDate,))
            connection.commit()
        except Exception as e:
            SendEmail(repDate, "error", f"Procedure Execution Failed: {e}")
            sys.exit(1)
        finally:
            cursor.close()
            connection.close()

  
        
        SendEmail(repDate,'success','Uploaded to Oracle Successfully')


        
    except Exception as e:
        error_msg = f"Failed Due to File Loading into Oracle: {e}"
        SendEmail(repDate,"error",error_msg)
        sys.exit(1) 

In [10]:
def RemoveFiles(repDate, local_paths):


    sftp_server = '...'
    sftp_username = '...'
    sftp_password = '...'
    
    transport = paramiko.Transport((sftp_server, 22))
        
    try:
        transport.connect(username=sftp_username, password=sftp_password)
        sftp = paramiko.SFTPClient.from_transport(transport)
        sftp.chdir(remote_directory)
        
        if isinstance(local_paths, (str, os.PathLike)):
            local_paths = [str(local_paths)]
        elif not isinstance(local_paths, (list, tuple)):
            local_paths = list(local_paths)
        
        expected_prefix = f"adl_{repDate}_"
        basenames = []
        for p in local_paths:
            bn = os.path.basename(p)
            if bn.startswith(expected_prefix) and bn.endswith(".csv"):
                basenames.append(bn)

        # remove from remote folder
        #for bn in basenames:
        #    remote_path = f"{remote_directory}{bn}"
        #    try:
        #        sftp.remove(remote_path)
        #    except Exception as e:
        #        SendEmail(repDate, "error", f"Failed to delete remote {remote_path}: {e}")

        # remove from local folder
        for p in local_paths:
            try:
                if os.path.exists(p):
                    os.remove(p)
        
            except Exception as e:
                SendEmail(repDate, "error", f"Failed to delete local {p}: {e}")    

    
    except Exception as e:
        SendEmail(repDate, "error", f"Failed to delete remote: {e}")
        
    finally:
        try:
            if sftp:
                sftp.close()
        except:
            pass
        if transport:
            transport.close()

In [ ]:
def main():
    argv_len = len(sys.argv)
    num_args = argv_len-1

    if num_args == 0:
        yesterday_d = datetime.now() - timedelta(days=1)
        run_date_str = yesterday_d.strftime('%Y%m%d')
    else:
        run_date_str = sys.argv[1]

    try:

        low_count_threshold = 1000
        date_column = 'DATA_DATE'
        remote_directory = '...'
        process_name = 'ADL'
        target_schema = '...'
        target_table = 'ADL_AJR_RAW'
    
        op1 = SFTPDownload(run_date_str)
        
        op2 = ProcessFile(run_date_str,op1)
        
        UploadToOracle(run_date_str,op2)
        
    except Exception as e:
        SendEmail(run_date_str, "error", f"Code run failed: {e}")
        sys.exit(1)
    
if __name__ == '__main__':
    main()